# Fine-tuning Whisper small on Hungarian (VoxPopuli)

COE025 Natural Language Processing, Project 3, Group 8.

Run it on Google Colab with a GPU: **Runtime > Change runtime type**, a GPU with more than 16 GB of memory (the committed run used an NVIDIA L4; a 15 GB T4 runs out of memory with 16 clips per update), then **Runtime > Run all**. No file has to be uploaded: the notebook clones the public repository, which holds the code and the audio.

What happens, in this order:

1. clone the repository, install the pinned packages, record the GPU, unpack the audio, fetch the pinned checkpoint;
2. a short trial run that checks the whole path;
3. **before:** `openai/whisper-small` as published, on the validation split (zero-shot baseline);
4. fine-tuning on the training split; the validation split only chooses which checkpoint is kept;
5. **after:** the kept checkpoint on the same validation clips, with the same runner, decoding, references, normalisation and scorer, and a paired comparison of the two runs;
6. the small result files are packed for the repository. Model weights stay on the Colab disk.

**The test split is not used anywhere in this notebook.** It is evaluated once, at the very end of the project, and the runner refuses it unless the flag for that final evaluation is given.

Every setting is fixed in `src/huasr/finetune.py` and `src/huasr/whisper_hf.py`; the cells only call that code, so every number comes from committed code. On the NVIDIA L4 of the committed run: about 10 minutes of preparation, about 22 minutes for each full validation evaluation with beam 5 and 63.5 minutes of fine-tuning including nine validation passes. The trial run prints a projection measured on your GPU.

## 1. Repository and packages

In [ ]:
# Clone the public repository: code and audio (about 1.3 GB). Nothing is uploaded by hand.
import os

REPOSITORY = "https://github.com/kaankalkan03/NLP-project.git"
BRANCH = "main"
FOLDER = "/content/NLP-project"

if not os.path.isdir(FOLDER):  # running this cell again keeps the existing clone
    !git clone --depth 1 --branch {BRANCH} {REPOSITORY} {FOLDER}
%cd {FOLDER}
!git log -1 --format="commit %H  %cd"

In [ ]:
# A failed command must not be followed by hours of work on top of it: this stops "Run all".
def stop_unless_done(step):
    assert globals().get("_exit_code", 0) == 0, f"{step} failed: read the message above before going on"

The versions in `requirements-train.txt` are those of the Colab GPU runtime, so `pip` keeps the installed GPU build of torch. `pip` may report that other Colab packages want different versions of numpy, pyarrow or requests; those packages are not used here. If Colab offers to restart the session, that is not needed: every step below runs in a new Python process. (After a restart, run the first two cells again.)

The fine-tuning code does not use the package `datasets`, but `transformers` imports it when it is installed. If the trial run in section 3 stops while importing `datasets`, run `!pip uninstall -y datasets` and repeat that cell.

In [ ]:
# Pinned packages: those of the repository, the training-only ones, then the package itself.
!pip install -q -r requirements.txt -r requirements-train.txt
!pip install -q --no-deps -e .

In [ ]:
# The GPU and the package versions; every run below writes them into its record.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -m huasr.whisper_hf environment

In [ ]:
# Unit tests of the fine-tuning code. No model is loaded.
!python -m unittest tests.test_finetune tests.test_finetune_run tests.test_whisper_hf tests.test_compare 2>&1 | tail -n 4

## 2. Data and checkpoint

`unpack` restores every clip of the repository from the tar shards and checks it against the manifest. The fine-tuning code reads only the folders `train` and `validation`.

In [ ]:
# Audio: FLAC files out of the tar shards, each checked against the SHA-256 in the manifest.
!python -m huasr.subset unpack --raw data/raw/voxpopuli
stop_unless_done("unpack")
# Checkpoint: openai/whisper-small at the pinned revision. This is the only model download.
!python -m huasr.whisper_hf fetch
stop_unless_done("fetch")

In [ ]:
# What the fine-tuning will use: clips per split, clips left out by each rule, the fixed settings.
!python -m huasr.finetune plan

## 3. Trial run (a few minutes)

Eight updates on 64 training clips at the real batch size, three small evaluations, save and reload. It shows that everything works and fits in the GPU memory before two hours are spent, and it writes only below `scratch/`. If it stops with an out-of-memory error, add `--batch-size 8` to this cell and to the fine-tuning cell: an update then takes two batches of 8 clips instead of one of 16.

In [ ]:
# Prints the time per update, the peak GPU memory and a rough projection for the full run.
!python -m huasr.finetune train --smoke
stop_unless_done("the trial run")

## 4. Before: zero-shot baseline on the validation split

The published checkpoint, one clip at a time, with language Hungarian, task transcribe, no timestamps, beam size 5. Result folder: `results/baselines/hf-whisper-small/validation/`.

In [ ]:
# The published checkpoint on every validation clip, then its scores.
!python -m huasr.evaluate --system hf-whisper-small --force
stop_unless_done("the zero-shot evaluation")
!python -m huasr.score results/baselines/hf-whisper-small/validation
stop_unless_done("scoring")

## 5. Fine-tuning (about 2 hours)

2,000 updates of 16 clips with learning rate 1e-5 after a warm-up, mixed precision, seed 0. Every 250 updates the validation clips are decoded and scored with the scoring code of the repository; the checkpoint with the lowest validation WER is kept. Keep the browser tab open. After an interruption in the same session, run the cell again: it continues from the last checkpoint. A finished run is not repeated; `--force` trains again from the start.

In [ ]:
# Writes the kept model to scratch/finetune/hf-whisper-small/best (not tracked) and the training record
# to results/own/hf-whisper-small-finetuned/training.json.
!python -m huasr.finetune train --resume
stop_unless_done("the fine-tuning")

In [ ]:
# Validation scores per evaluated step, from the training record (greedy decoding, used for selection only).
import json

record = json.load(open("results/own/hf-whisper-small-finetuned/training.json", encoding="utf-8"))
print("training clips:", record["data"]["train"]["clips"], "- left out:", record["data"]["train"]["excluded"])
for row in record["evaluations"]:
    print(f"step {row['step']:>5}  WER {row['wer']:6.2f}%  CER {row['cer']:6.2f}%  loss {row['loss']:.4f}")
print("kept: step", record["best"]["step"])

## 6. After: the fine-tuned model on the same clips

Same runner, same decoding and same scorer as in section 4. Result folder: `results/own/hf-whisper-small-finetuned/validation/`, separate from the baselines.

In [ ]:
# The kept checkpoint on every validation clip, then its scores.
!python -m huasr.evaluate --system hf-whisper-small-finetuned --force
stop_unless_done("the evaluation of the fine-tuned model")
!python -m huasr.score results/own/hf-whisper-small-finetuned/validation
stop_unless_done("scoring")

In [ ]:
# Before and after on the same clips: difference, 95% interval and p-value from a paired bootstrap over speakers.
!python -m huasr.compare results/own/hf-whisper-small-finetuned/validation results/baselines/hf-whisper-small/validation
# The result tables of the repository, written from the score files.
!python -m huasr.report
!sed -n '/generated: results-baselines/,/end generated/p; /generated: results-own/,/end generated/p' results/README.md

## 7. Result files for the repository

Predictions, run records, scores, the comparison and the training record are small text files. The cell packs them and starts the download. No weights are in the archive.

In [ ]:
# Pack the two result folders and download the archive.
ARCHIVE = "/content/finetune_results.zip"
!rm -f {ARCHIVE}
!zip -q -r {ARCHIVE} results/baselines/hf-whisper-small results/own/hf-whisper-small-finetuned
!unzip -l {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

Unpack the archive in the root of your own clone of the repository, then:

```bash
python -m huasr.report      # refresh the tables in README.md and results/README.md
python -m unittest          # includes the check that the READMEs are up to date
git add results/baselines/hf-whisper-small results/own/hf-whisper-small-finetuned results/README.md README.md
git commit -m "Add zero-shot and fine-tuned Whisper small on the validation split"
```

Commit from your own GitHub account. Never add the folder `scratch/` or a `model.safetensors` file to the repository.

## Optional: keep the weights on Google Drive

Off by default. The kept model is lost when the Colab session ends. Set `SAVE_TO_DRIVE = True` to copy it (about 1 GB) to your Drive; Colab then asks for access to the Drive. To evaluate the model in a later session, mount the Drive and set the environment variable `HUASR_FINETUNED_DIR` to that folder before calling `huasr.evaluate`.

In [ ]:
# OPTIONAL: copy the kept model (weights, tokenizer and feature extractor files, training record) to Google Drive.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/huasr/hf-whisper-small-finetuned
    !cp -r scratch/finetune/hf-whisper-small/best/. /content/drive/MyDrive/huasr/hf-whisper-small-finetuned/